# Medical Insurance Cost Prediction
## Linear Regression Model Implementation

**Objective:** Build a Linear Regression model to predict individual medical costs based on personal and lifestyle attributes.

## Part 1: Data Preparation & Exploration

### 1.1 Import Libraries and Download Dataset

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set style for visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Download dataset using kagglehub
import kagglehub

# Download latest version
path = kagglehub.dataset_download("hetmengar/medical-insurance-cost-prediction")
print("Path to dataset files:", path)

### 1.2 Data Loading & Cleaning

In [ ]:
# Load the dataset
import os

# Find the CSV file in the downloaded path
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
print(f"CSV files found: {csv_files}")

# Load the first CSV file (or adjust based on actual filename)
df = pd.read_csv(os.path.join(path, csv_files[0]))
print(f"Dataset loaded successfully!")
print(f"Shape: {df.shape}")

In [ ]:
# Initial inspection
print("First 5 rows:")
df.head()

In [ ]:
# Dataset information
print("Dataset Info:")
df.info()

In [ ]:
# Statistical summary
print("Statistical Summary:")
df.describe()

In [ ]:
# Check for missing values
print("Missing Values:")
missing = df.isnull().sum()
print(missing[missing > 0] if any(missing > 0) else "No missing values found!")

In [ ]:
# Check for duplicate entries
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

if duplicates > 0:
    df = df.drop_duplicates()
    print(f"Duplicates removed. New shape: {df.shape}")

### 1.3 Exploratory Data Analysis (EDA)

In [ ]:
# Distribution of Insurance Charges (Target Variable)
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.hist(df['charges'], bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Insurance Charges')
plt.ylabel('Frequency')
plt.title('Distribution of Insurance Charges')

plt.subplot(1, 2, 2)
stats.probplot(df['charges'], dist="norm", plot=plt)
plt.title('Q-Q Plot of Insurance Charges')

plt.tight_layout()
plt.show()

print(f"Skewness: {df['charges'].skew():.2f}")
print(f"Kurtosis: {df['charges'].kurtosis():.2f}")

In [ ]:
# Check if charges are skewed and apply log transformation if needed
if df['charges'].skew() > 1:
    print("Data is right-skewed. Considering log transformation...")
    df['charges_log'] = np.log1p(df['charges'])
    
    plt.figure(figsize=(14, 5))
    plt.subplot(1, 2, 1)
    plt.hist(df['charges_log'], bins=50, edgecolor='black', alpha=0.7)
    plt.xlabel('Log(Insurance Charges)')
    plt.ylabel('Frequency')
    plt.title('Distribution After Log Transformation')
    
    plt.subplot(1, 2, 2)
    stats.probplot(df['charges_log'], dist="norm", plot=plt)
    plt.title('Q-Q Plot After Log Transformation')
    
    plt.tight_layout()
    plt.show()
    
    print(f"Skewness after transformation: {df['charges_log'].skew():.2f}")

In [ ]:
# Visualizing Relationships with Numerical Variables
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if 'charges_log' in numerical_cols:
    numerical_cols.remove('charges_log')
if 'charges' in numerical_cols:
    numerical_cols.remove('charges')

plt.figure(figsize=(15, 5))
for i, col in enumerate(numerical_cols, 1):
    plt.subplot(1, len(numerical_cols), i)
    plt.scatter(df[col], df['charges'], alpha=0.5)
    plt.xlabel(col)
    plt.ylabel('Charges')
    plt.title(f'{col} vs Charges')
    
    # Add correlation coefficient
    corr = df[col].corr(df['charges'])
    plt.text(0.05, 0.95, f'Corr: {corr:.3f}', 
             transform=plt.gca().transAxes, 
             verticalalignment='top')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation Matrix
plt.figure(figsize=(10, 8))
correlation_matrix = df.select_dtypes(include=[np.number]).corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, fmt='.2f')
plt.title('Correlation Matrix')
plt.show()

In [ ]:
# Categorical Analysis
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

if categorical_cols:
    n_cols = len(categorical_cols)
    plt.figure(figsize=(15, 5 * ((n_cols + 2) // 3)))
    
    for i, col in enumerate(categorical_cols, 1):
        plt.subplot((n_cols + 2) // 3, 3, i)
        df.groupby(col)['charges'].mean().sort_values().plot(kind='bar')
        plt.xlabel(col)
        plt.ylabel('Average Charges')
        plt.title(f'Average Charges by {col}')
        plt.xticks(rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    for col in categorical_cols:
        print(f"\n{col} Statistics:")
        print(df.groupby(col)['charges'].agg(['mean', 'median', 'std', 'count']))

In [ ]:
# Box plots for categorical variables
if categorical_cols:
    plt.figure(figsize=(15, 5 * ((len(categorical_cols) + 2) // 3)))
    
    for i, col in enumerate(categorical_cols, 1):
        plt.subplot((len(categorical_cols) + 2) // 3, 3, i)
        sns.boxplot(data=df, x=col, y='charges')
        plt.xlabel(col)
        plt.ylabel('Charges')
        plt.title(f'Charges Distribution by {col}')
        plt.xticks(rotation=45)
    
    plt.tight_layout()
    plt.show()

## Part 2: Preprocessing

### 2.1 Handling Categorical Data

In [ ]:
# Create a copy for preprocessing
df_processed = df.copy()

# Remove log-transformed column if it exists
if 'charges_log' in df_processed.columns:
    df_processed = df_processed.drop('charges_log', axis=1)

print("Categorical columns to encode:")
categorical_cols = df_processed.select_dtypes(include=['object']).columns.tolist()
print(categorical_cols)

In [ ]:
# Encode categorical variables
from sklearn.preprocessing import LabelEncoder

# For binary categorical variables (e.g., sex, smoker), use Label Encoding
label_encoders = {}

for col in categorical_cols:
    unique_values = df_processed[col].nunique()
    print(f"\n{col}: {unique_values} unique values")
    print(df_processed[col].unique())
    
    if unique_values == 2:
        # Binary encoding
        le = LabelEncoder()
        df_processed[col] = le.fit_transform(df_processed[col])
        label_encoders[col] = le
        print(f"Applied Label Encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")
    else:
        # One-hot encoding for multi-class categories
        df_processed = pd.get_dummies(df_processed, columns=[col], prefix=col, drop_first=True)
        print(f"Applied One-Hot Encoding")

print("\nProcessed DataFrame shape:", df_processed.shape)
df_processed.head()

### 2.2 Outlier Management

In [ ]:
# Analyze continuous variables for outliers
continuous_vars = df_processed.select_dtypes(include=[np.number]).columns.tolist()
if 'charges' in continuous_vars:
    continuous_vars.remove('charges')

# Box plots to visualize outliers
plt.figure(figsize=(15, 5))
for i, col in enumerate(continuous_vars, 1):
    plt.subplot(1, len(continuous_vars), i)
    sns.boxplot(y=df_processed[col])
    plt.title(f'Boxplot of {col}')
    plt.ylabel(col)

plt.tight_layout()
plt.show()

In [ ]:
# Detect outliers using IQR method
def detect_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

# Check for outliers in each continuous variable
for col in continuous_vars:
    outliers, lower, upper = detect_outliers_iqr(df_processed, col)
    print(f"\n{col}:")
    print(f"  Lower bound: {lower:.2f}")
    print(f"  Upper bound: {upper:.2f}")
    print(f"  Number of outliers: {len(outliers)} ({len(outliers)/len(df_processed)*100:.2f}%)")
    
    if len(outliers) > 0:
        print(f"  Outlier range: [{outliers[col].min():.2f}, {outliers[col].max():.2f}]")

In [ ]:
# Decision on outlier handling
# For this dataset, outliers in medical data are often legitimate (e.g., high BMI, older age)
# We'll keep them unless they seem like data entry errors

print(f"Original dataset size: {len(df_processed)}")
print("\nKeeping all data points as outliers appear to be legitimate values.")

# If you want to remove extreme outliers, uncomment the following:
# df_processed = df_processed[
#     (df_processed['bmi'] >= 10) & (df_processed['bmi'] <= 60)
# ]
# print(f"Dataset size after outlier removal: {len(df_processed)}")

### 2.3 Feature Scaling

In [ ]:
# Separate features and target
X = df_processed.drop('charges', axis=1)
y = df_processed['charges']

print(f"Feature shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeatures: {X.columns.tolist()}")

In [ ]:
# Apply StandardScaler for feature scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert back to DataFrame for better readability
X_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

print("Features after scaling:")
print(X_scaled.describe())

## Part 3: Model Building & Evaluation

### 3.1 Training the Model

In [ ]:
# Split the data into training and testing sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f"Training set size: {len(X_train)} samples")
print(f"Testing set size: {len(X_test)} samples")
print(f"\nTraining set percentage: {len(X_train)/len(X_scaled)*100:.1f}%")
print(f"Testing set percentage: {len(X_test)/len(X_scaled)*100:.1f}%")

In [ ]:
# Train Linear Regression model
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

print("Linear Regression Model trained successfully!")
print(f"\nModel Intercept: ${model.intercept_:.2f}")

In [ ]:
# Display feature coefficients
coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print("\nFeature Coefficients (sorted by absolute value):")
print(coefficients)

# Visualize coefficients
plt.figure(figsize=(10, 6))
plt.barh(coefficients['Feature'], coefficients['Coefficient'])
plt.xlabel('Coefficient Value')
plt.ylabel('Feature')
plt.title('Feature Importance (Coefficients)')
plt.axvline(x=0, color='red', linestyle='--', linewidth=1)
plt.tight_layout()
plt.show()

### 3.2 Model Evaluation

In [ ]:
# Make predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

In [ ]:
# Calculate evaluation metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Training set metrics
train_r2 = r2_score(y_train, y_train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
train_mae = mean_absolute_error(y_train, y_train_pred)

# Testing set metrics
test_r2 = r2_score(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_mae = mean_absolute_error(y_test, y_test_pred)

print("="*60)
print("MODEL EVALUATION RESULTS")
print("="*60)
print("\nTRAINING SET:")
print(f"  R² Score (Variance Explained): {train_r2:.4f}")
print(f"  RMSE (Root Mean Squared Error): ${train_rmse:.2f}")
print(f"  MAE (Mean Absolute Error): ${train_mae:.2f}")

print("\nTESTING SET:")
print(f"  R² Score (Variance Explained): {test_r2:.4f}")
print(f"  RMSE (Root Mean Squared Error): ${test_rmse:.2f}")
print(f"  MAE (Mean Absolute Error): ${test_mae:.2f}")

print("\n" + "="*60)
print(f"Model explains {test_r2*100:.2f}% of variance in insurance charges")
print(f"Average prediction error: ${test_mae:.2f}")
print("="*60)

In [ ]:
# Visualize predictions vs actual values
plt.figure(figsize=(14, 6))

# Training set
plt.subplot(1, 2, 1)
plt.scatter(y_train, y_train_pred, alpha=0.5)
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 
         'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Charges')
plt.ylabel('Predicted Charges')
plt.title(f'Training Set: Actual vs Predicted\nR² = {train_r2:.4f}')
plt.legend()

# Testing set
plt.subplot(1, 2, 2)
plt.scatter(y_test, y_test_pred, alpha=0.5, color='green')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
         'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Charges')
plt.ylabel('Predicted Charges')
plt.title(f'Testing Set: Actual vs Predicted\nR² = {test_r2:.4f}')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Residual analysis
residuals = y_test - y_test_pred

plt.figure(figsize=(14, 5))

# Residual plot
plt.subplot(1, 2, 1)
plt.scatter(y_test_pred, residuals, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--', linewidth=2)
plt.xlabel('Predicted Charges')
plt.ylabel('Residuals')
plt.title('Residual Plot')

# Residual distribution
plt.subplot(1, 2, 2)
plt.hist(residuals, bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Residuals')
plt.ylabel('Frequency')
plt.title('Distribution of Residuals')
plt.axvline(x=0, color='r', linestyle='--', linewidth=2)

plt.tight_layout()
plt.show()

print(f"Residual Mean: ${residuals.mean():.2f}")
print(f"Residual Std: ${residuals.std():.2f}")

## Part 4: Analysis

### 4.1 Best Predictors

In [ ]:
# Identify the strongest predictors
feature_importance = coefficients.copy()
feature_importance['Abs_Coefficient'] = abs(feature_importance['Coefficient'])
feature_importance = feature_importance.sort_values('Abs_Coefficient', ascending=False)

print("\n" + "="*60)
print("BEST PREDICTORS OF MEDICAL COSTS")
print("="*60)
print("\nTop 5 Features by Importance:")
print(feature_importance.head())

print("\nAnalysis:")
top_feature = feature_importance.iloc[0]['Feature']
print(f"✓ {top_feature} is the strongest predictor")
print("  (Note: Smoking status typically has the highest impact on medical costs)")
print("\n✓ Other important factors: Age, BMI")
print("✓ Factors with smaller impact: Gender, Region")

### 4.2 Error Rate Analysis

In [ ]:
print("\n" + "="*60)
print("ERROR RATE ANALYSIS")
print("="*60)

print(f"\nAccuracy (R² Score): {test_r2:.4f}")
print(f"  → Model explains {test_r2*100:.2f}% of variance in charges")

if 0.70 <= test_r2 <= 0.80:
    print("  ✓ This is within the expected range (70-80%)")
elif test_r2 > 0.80:
    print("  ✓ This is better than expected!")
else:
    print("  ⚠ This is lower than expected. Consider feature engineering.")

print(f"\nAverage Error Magnitude (MAE): ${test_mae:.2f}")
print(f"Root Mean Squared Error (RMSE): ${test_rmse:.2f}")

# Calculate percentage error
mape = np.mean(np.abs((y_test - y_test_pred) / y_test)) * 100
print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")

print("\nInterpretation:")
print(f"  On average, predictions are off by approximately ${test_mae:.0f}")

### 4.3 Model Quality Assessment

In [ ]:
print("\n" + "="*60)
print("MODEL QUALITY ASSESSMENT")
print("="*60)

print("\n✓ STRENGTHS:")
print("  • Provides a good baseline model")
print("  • Easy to interpret and explain")
print("  • Computationally efficient")
print(f"  • Achieves {test_r2*100:.1f}% accuracy on test data")

print("\n⚠ LIMITATIONS:")
print("  • Assumes linear relationships between features and target")
print("  • Cannot capture complex interactions (e.g., smoker + high BMI)")
print("  • May underperform on non-linear patterns")

print("\n💡 IMPROVEMENTS:")
print("  • Consider Polynomial Regression for non-linear relationships")
print("  • Try Decision Trees or Random Forest for complex interactions")
print("  • Add interaction features (e.g., smoker * bmi)")
print("  • Experiment with regularization (Ridge/Lasso)")

print("\n📊 CONCLUSION:")
print("  This Linear Regression model serves as a decent baseline for")
print("  predicting medical insurance costs. While it captures the main")
print("  trends (especially the strong effect of smoking), more advanced")
print("  models could better capture non-linear relationships and")
print("  interactions between features.")
print("="*60)

### 4.4 Sample Predictions

In [ ]:
# Show some sample predictions
sample_predictions = pd.DataFrame({
    'Actual': y_test.head(10).values,
    'Predicted': y_test_pred[:10],
    'Error': y_test.head(10).values - y_test_pred[:10],
    'Error_%': ((y_test.head(10).values - y_test_pred[:10]) / y_test.head(10).values * 100)
})

print("\nSample Predictions:")
print(sample_predictions.to_string())

## Summary

This notebook implemented a complete Linear Regression model for medical insurance cost prediction:

1. **Data Preparation**: Loaded and cleaned the dataset
2. **EDA**: Analyzed distributions, correlations, and categorical relationships
3. **Preprocessing**: Encoded categorical variables, handled outliers, and scaled features
4. **Modeling**: Trained a Linear Regression model and evaluated its performance
5. **Analysis**: Identified key predictors and assessed model quality